### Imbalanced Data Handling

A data-level fix for class imbalance, distinct from focal loss's loss-level fix (see `loss-functions.ipynb`, section 7). Covers random oversampling, random undersampling, SMOTE.

## Random Oversampling and Undersampling

#### 0. Core idea

Toy setup: 8 majority-class examples, 2 minority-class examples, labels = [0]*8 + [1]*2.

Random oversampling: duplicate minority-class rows (with replacement) until the classes are balanced, or closer to it.
```
original: 8 class-0, 2 class-1
oversample class-1 by duplicating: 8 class-0, 8 class-1 (the 2 real rows each appear 4 times)
```
Downside: no new information, the model sees literally the same 2 minority rows repeated, real risk of overfitting to those 2 specific examples, since gradient-based models effectively see them 4x as often in every batch.

Random undersampling: drop majority-class rows until balanced.
```
original: 8 class-0, 2 class-1
undersample class-0 by dropping 6 of them: 2 class-0, 2 class-1
```
Downside: throws away 75% of the majority class data, potentially real, useful signal, especially painful when the majority class is not actually huge in absolute terms.

In [ ]:
import numpy as np

X = np.arange(10).reshape(-1, 1)
y = np.array([0]*8 + [1]*2)

# oversample: duplicate minority rows
minority_idx = np.where(y == 1)[0]
oversample_idx = np.concatenate([np.arange(10), np.tile(minority_idx, 3)])  # 3 extra copies each
print("after oversampling, class counts:", np.bincount(y[oversample_idx]))

# undersample: drop majority rows
majority_idx = np.where(y == 0)[0]
rng = np.random.default_rng(0)
keep_majority = rng.choice(majority_idx, size=2, replace=False)
undersample_idx = np.concatenate([keep_majority, minority_idx])
print("after undersampling, class counts:", np.bincount(y[undersample_idx]))

## SMOTE (Synthetic Minority Oversampling Technique)

#### 0. Core idea: synthetic points, not duplicates

Fixes random oversampling's "no new information" problem by generating genuinely NEW synthetic minority points, interpolated between real minority points and their nearest minority neighbors, rather than copying existing rows.

Formula: new_point = point_a + r * (point_b - point_a), where point_b is one of point_a's nearest minority-class neighbors, and r is a random fraction between 0 and 1.

Worked example, 2D, two real minority points that are nearest neighbors of each other: point_a=(1,1), point_b=(3,3). Generate one synthetic point with r=0.5:
```
new_point = (1,1) + 0.5 * ((3,3) - (1,1))
          = (1,1) + 0.5 * (2,2)
          = (1,1) + (1,1)
          = (2,2)
```
(2,2) is a genuinely new point, sitting exactly halfway along the line segment between the two real minority points, not a duplicate of either. A different random r would land somewhere else on that same segment, e.g. r=0.2 gives (1.4, 1.4), r=0.8 gives (2.6, 2.6).

In [ ]:
import numpy as np

point_a = np.array([1, 1])
point_b = np.array([3, 3])

for r in [0.2, 0.5, 0.8]:
    synthetic = point_a + r * (point_b - point_a)
    print(f"r={r}: synthetic point = {synthetic}")

#### 1. Practical notes

Still has a real limitation: interpolating between minority points only makes sense if the minority class occupies a genuinely coherent, continuous region of feature space, if the minority class is actually several distinct sub-patterns (e.g. several genuinely different fraud typologies lumped under one "fraud" label), SMOTE can generate synthetic points that fall in the gap BETWEEN two real sub-patterns, a region that does not represent any real minority behavior at all. Variants exist to address this (Borderline-SMOTE focuses synthesis near the decision boundary specifically, ADASYN adapts the number of synthetic points generated per region based on how hard that region is to classify), not covered in depth here.

Applied only to the TRAINING set, never to validation/test, synthesizing fake examples into your evaluation data would let the model be "tested" on interpolated points that never occurred in reality, an evaluation-side leakage equivalent to the encoding leakage covered in the CatBoost notebook, just a different mechanism.

In [ ]:
# requires: pip install imbalanced-learn (not installed in this environment)
# from imblearn.over_sampling import SMOTE
#
# smote = SMOTE(k_neighbors=1, random_state=42)
# X_resampled, y_resampled = smote.fit_resample(X, y)
# print("class counts after SMOTE:", np.bincount(y_resampled))
print("see commented block above, requires: pip install imbalanced-learn")